# Ayaan Qayyum
## Intro to Deep Learning: Part 1: KNN Implementation

In [1]:
import numpy as np  
from download_mnist import load
import time
from download_mnist import load  
from datetime import datetime

Here we load the MNIST dataset. 

In [2]:
x_train, y_train, x_test, y_test = load()
x_train = x_train.reshape(60000,28,28)
x_test  = x_test.reshape(10000,28,28)
x_train = x_train.astype(float)
x_test = x_test.astype(float)

Here we display details about the MNIST dataset we imported. 

In [3]:
# Display dataset details
print("MNIST Dataset Details:")
print("-" * 30)

# Training Data
print(f"Training Images: {x_train.shape} (Samples, Height, Width)")
print(f"Training Labels: {y_train.shape} (Samples)")

# Test Data
print(f"Test Images: {x_test.shape} (Samples, Height, Width)")
print(f"Test Labels: {y_test.shape} (Samples)")

# Data Type Information
print("\nData Type Information:")
print(f"x_train dtype: {x_train.dtype}, x_test dtype: {x_test.dtype}")
print(f"y_train dtype: {y_train.dtype}, y_test dtype: {y_test.dtype}")

# Min-Max Values
print("\nPixel Intensity Ranges:")
print(f"Train Images: Min={x_train.min()}, Max={x_train.max()}")
print(f"Test Images: Min={x_test.min()}, Max={x_test.max()}")

# Unique Labels in Dataset
print("\nUnique Labels in Training Set:", np.unique(y_train))
print("Unique Labels in Test Set:", np.unique(y_test))

# Example of a single image shape and label
print("\nSample Image Shape:", x_train[0].shape)
print("Sample Label:", y_train[0])

MNIST Dataset Details:
------------------------------
Training Images: (60000, 28, 28) (Samples, Height, Width)
Training Labels: (60000,) (Samples)
Test Images: (10000, 28, 28) (Samples, Height, Width)
Test Labels: (10000,) (Samples)

Data Type Information:
x_train dtype: float64, x_test dtype: float64
y_train dtype: uint8, y_test dtype: uint8

Pixel Intensity Ranges:
Train Images: Min=0.0, Max=255.0
Test Images: Min=0.0, Max=255.0

Unique Labels in Training Set: [0 1 2 3 4 5 6 7 8 9]
Unique Labels in Test Set: [0 1 2 3 4 5 6 7 8 9]

Sample Image Shape: (28, 28)
Sample Label: 5


Here are some functions necessary for the tracking of progress and the distance metric calculations. 

In [4]:
def l1(data, query):
    return np.sum(np.abs(data - query), axis=1)

def l2(data, query):
    return np.sqrt(np.sum((data - query)**2, axis=1))

def classify_image(newInput, dataSet, labels, k, dist):
    if dist == 'l1':
        distances = l1(dataSet, newInput)
    elif dist == 'l2':
        distances = l2(dataSet, newInput)
    
    sorted_distances = distances.argsort()
    nearest_labels = [labels[i] for i in sorted_distances[:k]]
    counts = np.bincount(nearest_labels)
    return np.argmax(counts)


# Wrapper function to track progress in parallel
def classify_with_progress(index, newInput, dataSet, labels, k, dist, progress_tracker):
    result = classify_image(newInput, dataSet, labels, k, dist)

    # Track progress safely using a lock
    with lock:
        progress_tracker[0] += 1  # Increment progress counter
        if progress_tracker[0] % 100 == 0:  # Print every 100 images
            print(f"--- {progress_tracker[0]} images classified ---")

    return result

Here is the core KNN classification function. 

In [5]:
from multiprocessing.pool import ThreadPool as Pool
from threading import Lock

lock = Lock()

def kNNClassify(newInput, dataSet, labels, k): 
    
    ########################
    # Input your code here #
    ########################
    
    dist = "l2"
    
    # Reshape for consistency
    newInput = newInput.reshape(newInput.shape[0], -1)
    dataSet = dataSet.reshape(dataSet.shape[0], -1)

    # Print start time
    start_time = datetime.now()
    print(f"--- Program started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')} ---")

    # Shared progress counter
    progress_tracker = [0]  # Using a list so it can be modified inside threads

    # Perform KNN classification with multithreading and live progress tracking
    with Pool() as pool:
        result = pool.starmap(
            classify_with_progress, 
            [(i, newInput[i], dataSet, labels, k, dist, progress_tracker) for i in range(len(newInput))]
        )

    return np.array(result)
    
    ####################
    # End of your code #
    ####################


Here we run on 1000 images and half the dataset. 

In [6]:
start_time = time.time()
NUM_IMAGES = 1000
TRAIN_LIMIT = 30000
outputlabels = kNNClassify(x_test[0:NUM_IMAGES], x_train[0:TRAIN_LIMIT], y_train[0:TRAIN_LIMIT], 10)
result = y_test[0:NUM_IMAGES] - outputlabels
result = (1 - np.count_nonzero(result)/len(outputlabels))
print ("---classification accuracy for knn on mnist: %s ---" %result)
print ("---execution time: %s seconds ---" % (time.time() - start_time))

--- Program started at: 2025-02-11 15:13:05 ---
--- 100 images classified ---
--- 200 images classified ---
--- 300 images classified ---
--- 400 images classified ---
--- 500 images classified ---
--- 600 images classified ---
--- 700 images classified ---
--- 800 images classified ---
--- 900 images classified ---
--- 1000 images classified ---
---classification accuracy for knn on mnist: 0.938 ---
---execution time: 15.876988887786865 seconds ---


Here we run on 5000 images and the entire dataset. 

In [7]:
start_time = time.time()
NUM_IMAGES = 5000
TRAIN_LIMIT = 60000
outputlabels = kNNClassify(x_test[0:NUM_IMAGES], x_train[0:TRAIN_LIMIT], y_train[0:TRAIN_LIMIT], 10)
result = y_test[0:NUM_IMAGES] - outputlabels
result = (1 - np.count_nonzero(result)/len(outputlabels))
print ("---classification accuracy for knn on mnist: %s ---" %result)
print ("---execution time: %s seconds ---" % (time.time() - start_time))

--- Program started at: 2025-02-11 15:13:21 ---
--- 100 images classified ---
--- 200 images classified ---
--- 300 images classified ---
--- 400 images classified ---
--- 500 images classified ---
--- 600 images classified ---
--- 700 images classified ---
--- 800 images classified ---
--- 900 images classified ---
--- 1000 images classified ---
--- 1100 images classified ---
--- 1200 images classified ---
--- 1300 images classified ---
--- 1400 images classified ---
--- 1500 images classified ---
--- 1600 images classified ---
--- 1700 images classified ---
--- 1800 images classified ---
--- 1900 images classified ---
--- 2000 images classified ---
--- 2100 images classified ---
--- 2200 images classified ---
--- 2300 images classified ---
--- 2400 images classified ---
--- 2500 images classified ---
--- 2600 images classified ---
--- 2700 images classified ---
--- 2800 images classified ---
--- 2900 images classified ---
--- 3000 images classified ---
--- 3100 images classified ---


Here we run on all the images on the entire dataset. 

In [8]:
start_time = time.time()
NUM_IMAGES = 10000
TRAIN_LIMIT = 60000
outputlabels = kNNClassify(x_test[0:NUM_IMAGES], x_train[0:TRAIN_LIMIT], y_train[0:TRAIN_LIMIT], 10)
result = y_test[0:NUM_IMAGES] - outputlabels
result = (1 - np.count_nonzero(result)/len(outputlabels))
print ("---classification accuracy for knn on mnist: %s ---" %result)
print ("---execution time: %s seconds ---" % (time.time() - start_time))

--- Program started at: 2025-02-11 15:15:54 ---
--- 100 images classified ---
--- 200 images classified ---
--- 300 images classified ---
--- 400 images classified ---
--- 500 images classified ---
--- 600 images classified ---
--- 700 images classified ---
--- 800 images classified ---
--- 900 images classified ---
--- 1000 images classified ---
--- 1100 images classified ---
--- 1200 images classified ---
--- 1300 images classified ---
--- 1400 images classified ---
--- 1500 images classified ---
--- 1600 images classified ---
--- 1700 images classified ---
--- 1800 images classified ---
--- 1900 images classified ---
--- 2000 images classified ---
--- 2100 images classified ---
--- 2200 images classified ---
--- 2300 images classified ---
--- 2400 images classified ---
--- 2500 images classified ---
--- 2600 images classified ---
--- 2700 images classified ---
--- 2800 images classified ---
--- 2900 images classified ---
--- 3000 images classified ---
--- 3100 images classified ---


## Intro to Deep Learning: HW1 Part 2: Linear Classifier

In [ ]:
import numpy as np  
from download_mnist import load
from datetime import datetime

Here we load the dataset for the linear classifier. The dimensions are different as compared to the KNN method. 

In [ ]:
# Load dataset
x_train, y_train, x_test, y_test = load()

# Reshape and preprocess
x_train = x_train.reshape(60000, 28*28)  # Flatten to (60000, 784)
x_test = x_test.reshape(10000, 28*28)    # Flatten to (10000, 784)
x_train = x_train.astype(float) / 255.0  # Normalize
x_test = x_test.astype(float) / 255.0    # Normalize

Here we examine details about the MNIST dataset. 

In [15]:
# Display dataset details
print("MNIST Dataset Details:")
print("-" * 30)

# Training Data
print(f"Training Images: {x_train.shape} (Samples, Height, Width)")
print(f"Training Labels: {y_train.shape} (Samples)")

# Test Data
print(f"Test Images: {x_test.shape} (Samples, Height, Width)")
print(f"Test Labels: {y_test.shape} (Samples)")

# Data Type Information
print("\nData Type Information:")
print(f"x_train dtype: {x_train.dtype}, x_test dtype: {x_test.dtype}")
print(f"y_train dtype: {y_train.dtype}, y_test dtype: {y_test.dtype}")

# Min-Max Values
print("\nPixel Intensity Ranges:")
print(f"Train Images: Min={x_train.min()}, Max={x_train.max()}")
print(f"Test Images: Min={x_test.min()}, Max={x_test.max()}")

# Unique Labels in Dataset
print("\nUnique Labels in Training Set:", np.unique(y_train))
print("Unique Labels in Test Set:", np.unique(y_test))

# Example of a single image shape and label
print("\nSample Image Shape:", x_train[0].shape)
print("Sample Label:", y_train[0])

MNIST Dataset Details:
------------------------------
Training Images: (60000, 784) (Samples, Height, Width)
Training Labels: (60000,) (Samples)
Test Images: (10000, 784) (Samples, Height, Width)
Test Labels: (10000,) (Samples)

Data Type Information:
x_train dtype: float64, x_test dtype: float64
y_train dtype: uint8, y_test dtype: uint8

Pixel Intensity Ranges:
Train Images: Min=0.0, Max=1.0
Test Images: Min=0.0, Max=1.0

Unique Labels in Training Set: [0 1 2 3 4 5 6 7 8 9]
Unique Labels in Test Set: [0 1 2 3 4 5 6 7 8 9]

Sample Image Shape: (784,)
Sample Label: 9


Here we compute the Softmax probabilities, the Cross Entropy Loss, and the Accuracy from the result. 

In [25]:
# Softmax function to calculate probabilities
def softmax(logits):
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))  # Stability trick
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

# Computer the cross-entropy loss. 
def cross_entropy_loss(probs, y_true):
    num_samples = y_true.shape[0]
    correct_log_probs = -np.log(probs[range(num_samples), y_true] + 1e-9)  # Add small value for stability
    return np.sum(correct_log_probs) / num_samples

# Compute the classification accuracy
def accuracy(probs, y_true):
    preds = np.argmax(probs, axis=1)
    return np.mean(preds == y_true)

Here we implement random search and prediction to find the best model randomly. 

In [ ]:
# Find the best W and b randomly
def random_search(x_train, y_train, num_classes, num_trials=1000):
    num_samples, input_size = x_train.shape
    best_acc = 0
    best_W = None
    best_b = None

    for _ in range(num_trials):
        # Generate random weights and bias
        W = np.random.randn(input_size, num_classes) * 0.01
        b = np.random.randn(1, num_classes) * 0.01

        # Compute logits and softmax probabilities
        logits = np.dot(x_train, W) + b
        probs = softmax(logits)

        # Compute accuracy
        acc = accuracy(probs, y_train)

        # Save best model
        if acc > best_acc:
            best_acc = acc
            best_W = W
            best_b = b

    print(f"Best Training Accuracy Found: {best_acc:.4f}")
    return best_W, best_b

# Predict labels with trained weights. 
def predict(x, W, b):
    logits = np.dot(x, W) + b
    probs = softmax(logits)
    return np.argmax(probs, axis=1)

Here is the main logic code that performs random search to find the best linear classifier. 

In [ ]:
num_classes = 10

# Perform random search to find the best parameters
W_best, b_best = random_search(x_train, y_train, num_classes, num_trials=5000)

# Evaluate on test set
test_logits = np.dot(x_test, W_best) + b_best
test_probs = softmax(test_logits)
test_acc = accuracy(test_probs, y_test)

print(f"Test Accuracy: {test_acc:.4f}")

Best Training Accuracy Found: 0.2801
Test Accuracy: 0.2807


Here, as an extra, I implement gradient descent instead of random search to find the best linear classifier. 

In [21]:
def gradient_descent(x_train, y_train, num_classes, learning_rate=0.1, epochs=1000, batch_size=256):
    """Performs softmax regression using gradient descent."""
    num_samples, input_size = x_train.shape

    # Initialize weights and bias
    W = np.random.randn(input_size, num_classes) * 0.01
    b = np.zeros((1, num_classes))

    for epoch in range(epochs):
        # Shuffle dataset
        indices = np.random.permutation(num_samples)
        x_train_shuffled, y_train_shuffled = x_train[indices], y_train[indices]

        for i in range(0, num_samples, batch_size):
            X_batch = x_train_shuffled[i:i + batch_size]
            y_batch = y_train_shuffled[i:i + batch_size]

            # Compute logits and softmax probabilities
            logits = np.dot(X_batch, W) + b
            probs = softmax(logits)

            # Compute gradient of loss w.r.t W and b
            grads = probs
            grads[range(len(y_batch)), y_batch] -= 1  # One-hot encoding derivative trick
            grads /= len(y_batch)

            dW = np.dot(X_batch.T, grads)
            dB = np.sum(grads, axis=0, keepdims=True)

            # Update weights
            W -= learning_rate * dW
            b -= learning_rate * dB

        # Compute loss and accuracy for tracking progress
        train_logits = np.dot(x_train, W) + b
        train_probs = softmax(train_logits)
        train_loss = cross_entropy_loss(train_probs, y_train)
        train_acc = accuracy(train_probs, y_train)

        # Print progress every 100 epochs
        if epoch % 100 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}/{epochs} - Loss: {train_loss:.4f} - Accuracy: {train_acc:.4f}")

    print(f"Final Training Accuracy: {train_acc:.4f}")
    return W, b


In [28]:
num_classes = 10

# Perform random search to find the best parameters
LIMIT = len(x_train) - 1
W_best, b_best = gradient_descent(x_train[:LIMIT], y_train[:LIMIT], num_classes)

# Evaluate on test set
test_logits = np.dot(x_test, W_best) + b_best
test_probs = softmax(test_logits)
test_acc = accuracy(test_probs, y_test)

print(f"Test Accuracy: {test_acc:.4f}")

Epoch 0/1000 - Loss: 0.6357 - Accuracy: 0.7809
Epoch 100/1000 - Loss: 0.4454 - Accuracy: 0.8367
Epoch 200/1000 - Loss: 0.3680 - Accuracy: 0.8709
Epoch 300/1000 - Loss: 0.4015 - Accuracy: 0.8589
Epoch 400/1000 - Loss: 0.3697 - Accuracy: 0.8681
Epoch 500/1000 - Loss: 0.3511 - Accuracy: 0.8777
Epoch 600/1000 - Loss: 0.3644 - Accuracy: 0.8720
Epoch 700/1000 - Loss: 0.3550 - Accuracy: 0.8763
Epoch 800/1000 - Loss: 0.3514 - Accuracy: 0.8770
Epoch 900/1000 - Loss: 0.3452 - Accuracy: 0.8789
Epoch 999/1000 - Loss: 0.3494 - Accuracy: 0.8765
Final Training Accuracy: 0.8765
Test Accuracy: 0.8413
